[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-06-model-flavors.ipynb#scrollTo=11a1b1c1)

---
# Day 6 · MLflow Models and Flavors (sklearn, PyFunc, ONNX)
**certified-journeys / mlflow-certified** · Learn session

> **Goal for today:** Log an sklearn model with a signature and input example, understand model flavors, load the model back for inference, and build a custom `pyfunc` model that pre-processes inputs before prediction.


In [ ]:
%pip install -q mlflow scikit-learn pandas numpy


## Step 1 · What are MLflow model flavors?

An **MLflow Model** is a directory with a standard structure. The `MLmodel` YAML file at its root lists one or more **flavors** — each flavor tells downstream tools how to load and serve the model.

```
my_model/
├── MLmodel          ← flavor declarations
├── model.pkl        ← sklearn-flavor artifact
├── requirements.txt
└── input_example.json
```

| Flavor | Framework | Load function |
|---|---|---|
| `sklearn` | scikit-learn | `mlflow.sklearn.load_model()` |
| `python_function` | Any Python | `mlflow.pyfunc.load_model()` |
| `xgboost` | XGBoost | `mlflow.xgboost.load_model()` |
| `pytorch` | PyTorch | `mlflow.pytorch.load_model()` |
| `onnx` | ONNX Runtime | `mlflow.onnx.load_model()` |

Every model has the `python_function` flavor — it's the **universal adapter** used by the MLflow serving REST API regardless of framework.


In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.pyfunc
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from mlflow.models import infer_signature

# Local tracking
mlflow.set_tracking_uri("./mlruns")
mlflow.set_experiment("day06-model-flavors")

# Data
data = load_breast_cancer()
X, y = pd.DataFrame(data.data, columns=data.feature_names), data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Features (as DataFrame): {list(X.columns[:5])} ...")
print(f"Train shape: {X_train.shape}")


**What just happened?**

- We load `X` as a **Pandas DataFrame** (not a numpy array) — MLflow's `infer_signature` will use the column names to build a typed input schema.
- `from mlflow.models import infer_signature` gives us the function that inspects training data and predictions to auto-build a model signature.
- **A DataFrame input means the model signature will include column names and types**, which the serving REST API uses to validate incoming JSON payloads.


## Step 2 · Log an sklearn model with signature and input example

A **model signature** specifies the schema of inputs and outputs — column names, types, and shape.  
An **input example** is a small sample of real training data stored alongside the model.

Both together enable:
1. **Request validation** — the serving API rejects malformed payloads automatically
2. **Documentation** — downstream users know exactly what format to send
3. **Registry compatibility** — required when deploying to Databricks Model Registry

```python
signature = infer_signature(X_train, y_pred)  # inspects real data shapes
mlflow.sklearn.log_model(model, "model", signature=signature, input_example=X_train.head())
```


In [ ]:
with mlflow.start_run(run_name="rf-with-signature") as run:
    # Train
    model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Build the signature: infer_signature(inputs, outputs)
    # inputs = training DataFrame (preserves column names and dtypes)
    # outputs = model predictions (array of 0/1)
    signature = infer_signature(X_train, y_pred)

    # Log params and metrics
    mlflow.log_param("n_estimators", model.n_estimators)
    mlflow.log_param("max_depth", model.max_depth)
    mlflow.log_metric("accuracy", acc)

    # Log the model with signature + input example
    # input_example stores 5 rows of real data in the artifact
    model_info = mlflow.sklearn.log_model(
        model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(),
    )

    rf_run_id  = run.info.run_id
    model_uri  = model_info.model_uri

print(f"Model URI: {model_uri}")
print(f"Accuracy:  {acc:.4f}")
print(f"Signature inputs:  {signature.inputs}")
print(f"Signature outputs: {signature.outputs}")


**What just happened?**

- `infer_signature(X_train, y_pred)` auto-detected 30 float64 input columns (named) and a long integer output.
- `input_example=X_train.head()` stored 5 rows as JSON in the artifact — viewable in the MLflow UI under the model artifact tab.
- **The `model_uri`** (`runs:/<run_id>/model`) is the address used to load this model anywhere that has access to the tracking server.
- The generated `MLmodel` file now lists both `sklearn` and `python_function` flavors.


## Step 3 · Inspect the MLmodel file

The `MLmodel` file is the single source of truth for how a model can be loaded and served.  
It's a YAML file stored at the root of every logged model artifact.

Let's read it directly from the local `mlruns/` directory to see the flavor declarations.


In [ ]:
import os
import yaml
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Download the MLmodel file from the run artifact
local_path = client.download_artifacts(rf_run_id, "model/MLmodel", ".")

with open(local_path) as f:
    mlmodel = yaml.safe_load(f)

print("=== MLmodel flavors ===")
for flavor_name, flavor_data in mlmodel.get("flavors", {}).items():
    print(f"\n[{flavor_name}]")
    for k, v in flavor_data.items():
        # Truncate long lists for display
        display_v = str(v)[:80] + "..." if len(str(v)) > 80 else v
        print(f"  {k}: {display_v}")

print("\n=== Signature ===")
if "signature" in mlmodel:
    import json
    sig = mlmodel["signature"]
    inputs = json.loads(sig.get("inputs", "[]"))
    print(f"  Input columns ({len(inputs)}):")
    for col in inputs[:3]:  # show first 3
        print(f"    {col}")
    print(f"    ... ({len(inputs) - 3} more)")
    outputs = json.loads(sig.get("outputs", "[]"))
    print(f"  Output: {outputs}")


**What just happened?**

- The `MLmodel` YAML has two top-level sections: `flavors` and `signature`.
- **`python_function` flavor** — always present; specifies the entry point (`predict`) and `env` (conda.yaml).
- **`sklearn` flavor** — specifies the pickle path (`model.pkl`) and the sklearn version used at log time.
- The signature `inputs` is a JSON array of `{"name": "...", "type": "double"}` objects — exactly what the REST serving API uses for validation.


## Step 4 · Load the model back and run inference

MLflow provides two load patterns:

| Pattern | Function | Returns | Use when |
|---|---|---|---|
| Framework-specific | `mlflow.sklearn.load_model(uri)` | sklearn estimator | You need `.predict_proba()`, `.feature_importances_` |
| Universal pyfunc | `mlflow.pyfunc.load_model(uri)` | pyfunc wrapper | Production serving, framework-agnostic code |

Both accept the same `model_uri` — either `runs:/<run_id>/model` or a local path.


In [ ]:
# Load as sklearn model — returns the original RandomForestClassifier
loaded_sklearn = mlflow.sklearn.load_model(model_uri)
y_pred_sklearn = loaded_sklearn.predict(X_test)
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

print(f"Loaded sklearn model type: {type(loaded_sklearn).__name__}")
print(f"Accuracy (sklearn load):   {acc_sklearn:.4f}")
print(f"Feature importances (top 3):")
importances = pd.Series(
    loaded_sklearn.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)
for feat, imp in importances.head(3).items():
    print(f"  {feat}: {imp:.4f}")

print()

# Load as pyfunc — framework-agnostic; .predict() is the only method
loaded_pyfunc = mlflow.pyfunc.load_model(model_uri)
y_pred_pyfunc = loaded_pyfunc.predict(X_test)  # accepts DataFrame or numpy
acc_pyfunc = accuracy_score(y_test, y_pred_pyfunc)

print(f"Loaded pyfunc model type:  {type(loaded_pyfunc).__name__}")
print(f"Accuracy (pyfunc load):    {acc_pyfunc:.4f}")
print(f"Predictions match: {np.array_equal(y_pred_sklearn, y_pred_pyfunc)}")


**What just happened?**

- `mlflow.sklearn.load_model()` returns the **original Python object** — you have full access to `.feature_importances_`, `.predict_proba()`, etc.
- `mlflow.pyfunc.load_model()` returns a **`PyFuncModel` wrapper** — only exposes `.predict()`, but works for any flavor.
- **Predictions are identical** — both paths load the same underlying model; the difference is the Python interface.
- In production serving (`mlflow models serve`), MLflow always uses the `python_function` flavor internally — the pyfunc load pattern is the closest analog to how the REST API works.


## Step 5 · Build a custom `pyfunc` model with pre-processing

A **custom pyfunc** lets you bundle arbitrary Python logic (preprocessing, postprocessing, ensembles) into an MLflow model — not just a bare estimator.

To create one:
1. Subclass `mlflow.pyfunc.PythonModel`
2. Implement `predict(self, context, model_input)` — `context` gives access to artifacts
3. Log with `mlflow.pyfunc.log_model(python_model=..., artifacts={...})`

This is the pattern used for:
- Wrapping any non-standard framework (statsmodels, prophet, custom algorithms)
- Bundling a feature transformer + model as one deployable unit
- Adding business logic (class-name lookup, output rounding) to prediction


In [ ]:
import pickle
from sklearn.preprocessing import StandardScaler

# Train a scaler + model separately so we can bundle both in pyfunc
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

base_model = RandomForestClassifier(n_estimators=80, max_depth=4, random_state=42)
base_model.fit(X_train_scaled, y_train)

# Save artifacts to disk so pyfunc can reference them
scaler_path = "/tmp/scaler.pkl"
model_path  = "/tmp/base_model.pkl"
with open(scaler_path, "wb") as f: pickle.dump(scaler, f)
with open(model_path,  "wb") as f: pickle.dump(base_model, f)

# Define the custom pyfunc model class
class ScaledRandomForest(mlflow.pyfunc.PythonModel):
    """Pyfunc wrapper that applies StandardScaler before prediction."""

    def load_context(self, context):
        # context.artifacts maps names to local paths of logged artifacts
        with open(context.artifacts["scaler"], "rb") as f:
            self.scaler = pickle.load(f)
        with open(context.artifacts["model"], "rb") as f:
            self.model = pickle.load(f)

    def predict(self, context, model_input):
        # model_input is a DataFrame (if signature is set) or numpy array
        X_scaled = self.scaler.transform(model_input)
        predictions = self.model.predict(X_scaled)
        # Return as a DataFrame for clear column naming
        return pd.DataFrame({"prediction": predictions})


with mlflow.start_run(run_name="pyfunc-scaled-rf") as run:
    # Build signature using unscaled input (caller passes raw features)
    y_pred_test = base_model.predict(X_test_scaled)
    sig = infer_signature(
        X_test,                               # unscaled — what callers send
        pd.DataFrame({"prediction": y_pred_test})  # output DataFrame
    )

    mlflow.pyfunc.log_model(
        artifact_path="scaled_model",
        python_model=ScaledRandomForest(),
        artifacts={"scaler": scaler_path, "model": model_path},
        signature=sig,
        input_example=X_test.head(3),
    )

    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred_test))
    pyfunc_run_id = run.info.run_id

print(f"Custom pyfunc run ID: {pyfunc_run_id[:8]}...")


**What just happened?**

- `load_context` is called once when the model is loaded — it reads artifacts from the artifact store.
- `predict` receives **unscaled inputs** from the caller and applies the scaler internally — the caller doesn't need to know about preprocessing.
- **`artifacts` dict** maps logical names (e.g. `"scaler"`) to local file paths; MLflow copies these files into the run artifact store.
- The signature on the pyfunc uses **raw (unscaled) inputs** — matching what a real API caller would send.


## Step 6 · Load and test the custom pyfunc model

Loading a custom pyfunc is identical to loading any other MLflow model — the framework difference is fully abstracted.


In [ ]:
pyfunc_uri = f"runs:/{pyfunc_run_id}/scaled_model"

# Load the custom pyfunc model — identical API to loading any MLflow model
loaded_custom = mlflow.pyfunc.load_model(pyfunc_uri)

# Pass unscaled inputs — the model handles scaling internally
results = loaded_custom.predict(X_test)

print(f"Loaded model type: {type(loaded_custom).__name__}")
print(f"Output type:       {type(results).__name__}")
print(f"Output columns:    {list(results.columns)}")
print(f"Sample predictions:\n{results.head()}")

# Verify accuracy
acc = accuracy_score(y_test, results["prediction"])
print(f"\nAccuracy (custom pyfunc): {acc:.4f}")

# Compare with the raw sklearn load
print(f"Accuracy (direct sklearn):  {accuracy_score(y_test, base_model.predict(X_test_scaled)):.4f}")


**What just happened?**

- Loading the pyfunc called `load_context` once, which restored the scaler and model from the logged artifacts.
- **The caller passed unscaled `X_test`** — the pyfunc internally scaled it before prediction.
- Output is a DataFrame with a `prediction` column — matching the output schema declared in the signature.
- Accuracy matches the direct sklearn call (on pre-scaled data) — confirming the pyfunc's preprocessing is correct.


## Step 7 · Confirm the MLmodel file lists both flavors

Let's inspect the `MLmodel` for the custom pyfunc to confirm it only has the `python_function` flavor (no `sklearn` flavor, since we didn't use `mlflow.sklearn.log_model`).


In [ ]:
pyfunc_mlmodel_path = client.download_artifacts(
    pyfunc_run_id, "scaled_model/MLmodel", "."
)

with open(pyfunc_mlmodel_path) as f:
    pyfunc_mlmodel = yaml.safe_load(f)

print("=== Custom pyfunc MLmodel ===")
print(f"Flavors: {list(pyfunc_mlmodel.get('flavors', {}).keys())}")
print()

pf_flavor = pyfunc_mlmodel["flavors"].get("python_function", {})
print("python_function flavor details:")
for k, v in pf_flavor.items():
    print(f"  {k}: {v}")

print()
print("Comparing flavors by model type:")
print(f"  sklearn log_model:   flavors = ['sklearn', 'python_function']")
print(f"  pyfunc log_model:    flavors = {list(pyfunc_mlmodel.get('flavors', {}).keys())}")


**What just happened?**

- The custom pyfunc has only the `python_function` flavor — it has no framework-specific flavor since we used `mlflow.pyfunc.log_model()`.
- The `loader_module` in the pyfunc flavor points to `mlflow.pyfunc.model` — the entry point that calls `load_context()` then `predict()`.
- **Important**: `mlflow models serve` works with any MLflow model because it always uses the `python_function` flavor — your serving code doesn't need to know whether the underlying model is sklearn, XGBoost, or a custom class.


In [ ]:
# Challenge: Build and log your own custom pyfunc model
#
# 1. Create a pyfunc model class called 'ThresholdClassifier' that:
#    - Loads a trained RandomForestClassifier (train it in this cell)
#    - In predict(), calls .predict_proba() and applies a custom threshold of 0.6
#      (i.e. predict class=1 only if prob > 0.6, else class=0)
#    - Returns a DataFrame with columns 'probability' and 'prediction'
#
# 2. Log this model with:
#    - A signature inferred from X_test (input) and the output DataFrame
#    - input_example=X_test.head(3)
#
# 3. Load it back with mlflow.pyfunc.load_model() and compare accuracy
#    at threshold=0.6 vs the default threshold=0.5.
#    Hint: accuracy_score(y_test, loaded.predict(X_test)["prediction"])

# Your solution here:
# class ThresholdClassifier(mlflow.pyfunc.PythonModel):
#     def load_context(self, context): ...
#     def predict(self, context, model_input): ...


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| MLmodel file | YAML at artifact root; lists all flavors + signature |
| Model flavors | `sklearn` and `python_function` are the most common; pyfunc is the universal adapter |
| `infer_signature` | Pass `(X_train, y_pred)` to auto-detect input/output schema |
| `input_example` | Stores sample rows with the model — enables REST API payload docs |
| `mlflow.sklearn.load_model()` | Returns the raw sklearn estimator — full API access |
| `mlflow.pyfunc.load_model()` | Returns a `PyFuncModel` wrapper — `.predict()` only, but universal |
| Custom pyfunc | Subclass `PythonModel`, implement `load_context` + `predict` — bundle any logic |

> **Tip:** Always log a model signature and input example — they enable the model serving REST API to validate request payloads automatically.

---
## What's next
**Day 7** → MLflow Model Registry — register models, manage versions (Staging/Production/Archived), and set up approval workflows for model promotion.

Mark Day 6 complete in your [tracker](../index.html).
